# Week 4 Exercise Solution - Code Analyser and Reviewer

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
hf_token = os.getenv("HF_TOKEN")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")

In [ ]:
# Connect to client
openrouter_url = "https://openrouter.ai/api/v1"

def get_client():
  openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
  return openrouter

In [ ]:
# Models
models = [
  "gpt-4.1-mini", 
  "anthropic/claude-haiku-4.5", 
  "gemini-2.5-pro", 
  "openai/gpt-oss-120b", 
  "qwen2.5-coder", 
  "deepseek-coder-v2", 
  "gpt-oss:20b"
]

In [ ]:
# Prompts
SYSTEM_PROMPT = """You are an expert code reviewer and static analyzer. Your task is to analyze code across multiple dimensions and produce a single, structured response.

## Analysis Dimensions (inspect the code for ALL of these):

1. **Vulnerabilities** — SQL injection, XSS, hardcoded secrets, insecure deserialization, buffer overflows (C/C++), unsafe eval/exec, path traversal, command injection, insecure randomness.

2. **Optimization** — Time/space complexity, redundant loops, N+1 queries, unnecessary allocations, inefficient data structures/algorithms, redundant computations.

3. **Syntax & Correctness** — Parsing errors, type mismatches, logic errors, unhandled exceptions, off-by-one errors, null/undefined handling, division by zero, index out of bounds.

4. **Readability** — Structure, formatting, comments, clarity, excessive complexity.

5. **Naming** — Variable, function, and class names; adherence to language conventions and clarity.

6. **DRY (Don't Repeat Yourself)** — Duplicate logic, copy-paste, repeated patterns that could be extracted.

7. **Error Handling** — Exception handling adequacy, failure paths, validation of inputs, user-facing error clarity.

8. **Testability** — Dependency injection, side effects, mocking difficulty, clear contracts, coupling.

## Output Format — You MUST respond with exactly these 10 sections in this exact order, using this structure:

## Vulnerabilities
[Findings with format: **{Finding}**: {Description} — Line(s): {X} — Severity: {Critical|High|Medium|Low}. If none: "None identified."]

## Optimization
[Findings with format: **{Issue}**: {Description} — Line(s): {X} — Suggestion: {Brief fix}. If none: "No issues found."]

## Syntax & Correctness
[Findings with format: **{Issue}**: {Description} — Line(s): {X}. If none: "No issues found."]

## Readability
[Findings with format: **{Observation}**: {Description}. If none: "No issues found."]

## Naming
[Findings with format: **{Poor name}** → Suggested: **{Better name}** — Line(s): {X}. If none: "No issues found."]

## DRY (Don't Repeat Yourself)
[Findings with format: **{Duplicate logic}**: {Description} — Lines: {X, Y} — Suggestion: {Extract to function/module}. If none: "No issues found."]

## Error Handling
[Findings with format: **{Issue}**: {Description} — Line(s): {X} — Suggestion: {How to handle}. If none: "No issues found."]

## Testability
[Findings with format: **{Observation}**: {Description} — Impact: {Why it hinders testing}. If none: "No issues found."]

## Recommendations
[Numbered, prioritized list of improvements — security/correctness first, then optimization, then style.]

## Suggested Test Cases
[Generate 3–5 runnable unit tests in a fenced code block. Use the standard testing framework for the code's language (e.g., pytest for Python, jest for JavaScript, JUnit for Java). Cover: happy path, edge cases, and error conditions. Output format: ```{language}\n{test code}\n```]

## Rules
- **Language Verification**: Before analyzing, verify that the code syntax matches the language the user has selected (stated in the request). If the code clearly uses syntax from a different language (e.g., `def` and colons suggest Python but JavaScript was selected, or `function` and braces suggest JavaScript but Python was selected), respond with ONLY this message—replace {LANG} with the user's selected language:
  **⚠️ Language mismatch detected.** The provided code does not appear to be {LANG}. Please select the correct language from the dropdown or paste code that matches your selection.
  Do not perform the full analysis when a mismatch is detected; output only the mismatch message.
- Infer the language from the user's code and use it for the test block when performing analysis.
- Use line numbers when you can identify locations.
- Be specific and actionable; avoid generic advice.
- If a section has no findings, use exactly "None identified." or "No issues found." as specified above.
- Do not add sections or omit any of the 10 sections.
- Provide the full response in one reply.
"""

def make_user_prompt(user_code: str, language: str) -> str:
    return (
        "The user has selected **{language}** as the code language. "
        "First verify the code syntax matches {language}; if not, return the language mismatch message. "
        "Otherwise, review the following {language} code and provide the full analysis "
        "(Vulnerabilities, Optimization, Syntax & Correctness, Readability, Naming, DRY, "
        "Error Handling, Testability, Recommendations, and Suggested Test Cases).\n\n"
        "```{language}\n"
        "{user_code}\n"
        "```"
    ).format(language=language, user_code=user_code)

In [ ]:
# Analyze code
  
def analyze_code(code, language, model):
  user_prompt = make_user_prompt(code, language)
  messages = [
    {"role": "system", "content": SYSTEM_PROMPT}, 
    {"role": "user", "content": user_prompt}
  ]
  response = get_client().chat.completions.create(model=model, messages=messages)
  return response.choices[0].message.content

In [ ]:
# Gradio UI
DEFAULT_CODE_SAMPLE = """import sqlite3

def get_user(id):
    conn = sqlite3.connect("db.sqlite")
    cur = conn.cursor()
    cur.execute("SELECT * FROM users WHERE id=" + str(id))
    return cur.fetchone()

API_KEY = "sk-1234567890abcdef"

def process(x):
    return eval(x)

def calc_total(items):
    t = 0
    for i in range(len(items)):
        t = t + items[i]
    return t / (len(items) - 1)"""

LANGUAGES = [
    "Python", "JavaScript", "TypeScript", "Java", "Go", "Rust", "C", "C++",
    "C#", "Ruby", "PHP", "Swift", "Kotlin", "Scala"
]

CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;500;600&family=DM+Sans:wght@400;500;600;700&display=swap');

.gradio-container {
  font-family: 'DM Sans', sans-serif !important;
  max-width: 100% !important;
  padding: 1rem 2rem !important;
}

.gradio-container .dark {
  --background-fill-primary: #0f1117;
  --background-fill-secondary: #161b22;
  --block-background-fill: #161b22 !important;
  --border-color-primary: #30363d;
}

/* Header */
.analyzer-header {
  background: linear-gradient(135deg, #1a1f2e 0%, #252b3b 50%, #1e2433 100%);
  border: 1px solid #30363d;
  border-radius: 16px;
  padding: 1.5rem 2rem;
  margin-bottom: 1.5rem;
  box-shadow: 0 4px 24px rgba(0,0,0,0.3);
}

.analyzer-header h1 {
  font-family: 'DM Sans', sans-serif !important;
  font-size: 1.75rem !important;
  font-weight: 700 !important;
  color: #e6edf3 !important;
  margin: 0 0 0.25rem 0 !important;
  letter-spacing: -0.02em;
}

.analyzer-header p {
  font-size: 0.95rem !important;
  color: #8b949e !important;
  margin: 0 !important;
}

/* Code input card */
.code-panel {
  background: #161b22 !important;
  border: 1px solid #30363d !important;
  border-radius: 14px !important;
  padding: 1rem !important;
  box-shadow: 0 2px 12px rgba(0,0,0,0.2);
}

/* Analysis output card - fixed height and scrollable */
.analysis-panel {
  background: linear-gradient(180deg, #161b22 0%, #0d1117 100%) !important;
  border: 1px solid #30363d !important;
  border-radius: 14px !important;
  padding: 1.25rem !important;
  box-shadow: 0 2px 12px rgba(0,0,0,0.2);
  max-height: 70vh !important;
  overflow-y: auto !important;
}

.analysis-panel .prose, .analysis-panel .markdown {
  font-size: 0.9rem !important;
  line-height: 1.65 !important;
}

/* Analyze button */
.analyze-btn button {
  background: linear-gradient(135deg, #238636 0%, #2ea043 100%) !important;
  color: white !important;
  font-weight: 600 !important;
  border: none !important;
  border-radius: 10px !important;
  padding: 0.6rem 1.5rem !important;
  box-shadow: 0 2px 8px rgba(35, 134, 54, 0.4);
  transition: transform 0.15s, box-shadow 0.15s !important;
}

.analyze-btn button:hover {
  transform: translateY(-1px);
  box-shadow: 0 4px 14px rgba(35, 134, 54, 0.5) !important;
}

/* Controls row */
.controls-row .wrap {
  gap: 0.75rem;
  flex-wrap: wrap;
}

/* Dropdowns */
.gr-dropdown {
  border-radius: 10px !important;
  border-color: #30363d !important;
}

/* Block labels */
.block-label {
  font-weight: 600 !important;
  color: #8b949e !important;
}
"""

def run_analysis(code: str, language: str, model: str) -> str:
    """Wrapper for Gradio that handles empty input and errors."""
    if not code or not code.strip():
        return "*Please enter code to analyze.*"
    try:
        return analyze_code(code, language, model)
    except Exception as e:
        return f"**Error:** {str(e)}"

with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="emerald"), title="Code Analyser & Reviewer") as ui:
    gr.HTML("""
    <div class="analyzer-header">
      <h1>🔍 Code Analyser & Reviewer</h1>
      <p>Analyze code for vulnerabilities, optimization, syntax, readability, and more. Get actionable recommendations and test cases.</p>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            code_input = gr.Code(
                label="Code",
                value=DEFAULT_CODE_SAMPLE,
                language="python",
                lines=18,
                elem_classes=["code-panel"]
            )
            with gr.Row(elem_classes=["controls-row"]):
                language_dd = gr.Dropdown(LANGUAGES, value="Python", label="Language")
                model_dd = gr.Dropdown(models, value=models[0], label="Model")
                analyze_btn = gr.Button("Analyze Code", variant="primary", elem_classes=["analyze-btn"])

        with gr.Column(scale=1):
            analysis_output = gr.Markdown(
                value="*Your analysis will appear here.*",
                elem_classes=["analysis-panel"],
                label="Analysis"
            )

    analyze_btn.click(
        fn=run_analysis,
        inputs=[code_input, language_dd, model_dd],
        outputs=analysis_output
    )

ui.launch(inbrowser=True)